# Sales Variance Analysis

The aim is to find how much of the sales change comes from prices. The calculation starts with each customer and product, then adds the results by product, customer, and in total.

## 1. Load the data

Keep the CSV in the same folder as this notebook. The data contains 100 customers, five products, and two years. It is synthetic data, with amounts in EUR and no fixed charges.

In [1]:
import pandas as pd
from IPython.display import display

pd.options.display.float_format = '{:,.2f}'.format
df_sales = pd.read_csv('sales_variance_dataset.csv', encoding='utf-8-sig')
df_sales.head(10)


,Year,Customer,Product,Quantity,Price,Sales
0,2025,Customer_001,Product_A,272,4.60,"1,251.20"
1,2025,Customer_001,Product_B,243,7.58,"1,841.94"
2,2025,Customer_001,Product_C,76,11.04,839.04
3,2025,Customer_001,Product_D,406,16.89,"6,857.34"
4,2025,Customer_001,Product_E,363,23.00,"8,349.00"
5,2025,Customer_002,Product_A,479,5.20,"2,490.80"
6,2025,Customer_002,Product_B,351,8.16,"2,864.16"
7,2025,Customer_002,Product_C,916,12.36,"11,321.76"
8,2025,Customer_002,Product_D,874,18.18,"15,889.32"
9,2025,Customer_002,Product_E,943,25.50,"24,046.50"


In [2]:
print('Rows:', len(df_sales))
print('Customers:', df_sales['Customer'].nunique())
print('Products:', df_sales['Product'].nunique())
print('Years:', sorted(df_sales['Year'].unique().tolist()))


Rows: 1000
Customers: 100
Products: 5
Years: [2025, 2026]


## 2. Compare the two years

Put the 2025 and 2026 records side by side, matching on customer and product. All pairs in this dataset are established business and appear in both years.

In [3]:
previous_year = 2025
current_year = 2026
columns = ['Customer', 'Product', 'Quantity', 'Price', 'Sales']

# Each customer-product pair must have one record in each year.
if previous_year == current_year:
    raise ValueError('Choose two different years.')
if df_sales.duplicated(['Year', 'Customer', 'Product']).any():
    raise ValueError('There is more than one record for a customer-product pair in the same year.')

previous = df_sales.loc[df_sales['Year'] == previous_year, columns]
current = df_sales.loc[df_sales['Year'] == current_year, columns]
comparison = previous.merge(
    current, on=['Customer', 'Product'], how='outer',
    suffixes=('_Previous', '_Current'), validate='one_to_one'
)
if comparison.empty or comparison.isna().any().any():
    raise ValueError('Each customer-product pair needs complete data for both years.')

print('Customer-product pairs:', len(comparison))


Customer-product pairs: 500


## 3. Calculate the price impact

Use the change in price and the previous quantity:

$$\text{Price impact}=(P_{2026}-P_{2025})\times Q_{2025}$$

For example, a price increase from EUR 10 to EUR 11 on a previous quantity of 100 units gives a EUR 100 price impact. Keeping quantity at its old level isolates the price change. Positive values add to revenue; negative values reduce it.

In [4]:
comparison['Price_Change'] = (
    comparison['Price_Current'] - comparison['Price_Previous']
).round(2)
comparison['Price_Impact'] = (
    comparison['Price_Change'] * comparison['Quantity_Previous']
).round(2)

detail_columns = ['Customer', 'Product', 'Quantity_Previous',
                  'Price_Previous', 'Price_Current', 'Price_Change', 'Price_Impact']
comparison[detail_columns].head(10)


,Customer,Product,Quantity_Previous,Price_Previous,Price_Current,Price_Change,Price_Impact
0,Customer_001,Product_A,272,4.60,4.60,0.00,0.00
1,Customer_001,Product_B,243,7.58,6.97,-0.61,-148.23
2,Customer_001,Product_C,76,11.04,11.81,0.77,58.52
3,Customer_001,Product_D,406,16.89,15.88,-1.01,-410.06
4,Customer_001,Product_E,363,23.00,26.91,3.91,"1,419.33"
5,Customer_002,Product_A,479,5.20,5.36,0.16,76.64
6,Customer_002,Product_B,351,8.16,7.51,-0.65,-228.15
7,Customer_002,Product_C,916,12.36,14.58,2.22,"2,033.52"
8,Customer_002,Product_D,874,18.18,17.45,-0.73,-638.02
9,Customer_002,Product_E,943,25.50,27.29,1.79,"1,687.97"


## 4. Add up the results

For a product, add its price impacts across customers. For a customer, add the impacts from the products they bought. Both views give the same overall total.

The customer table below shows the first ten rows. `customer_summary` contains all 100 customers.

In [5]:
product_summary = comparison.groupby('Product', as_index=False)['Price_Impact'].sum().round(2)
customer_summary = comparison.groupby('Customer', as_index=False)['Price_Impact'].sum().round(2)
total_price_impact = round(comparison['Price_Impact'].sum(), 2)

print(f'Total price impact: EUR {total_price_impact:,.2f}')
display(product_summary)
display(customer_summary.head(10))


Total price impact: EUR 143,231.87


,Product,Price_Impact
0,Product_A,"5,706.92"
1,Product_B,"-8,227.50"
2,Product_C,"51,566.61"
3,Product_D,"4,994.35"
4,Product_E,"89,191.49"


,Customer,Price_Impact
0,Customer_001,919.56
1,Customer_002,"2,931.96"
2,Customer_003,"2,303.94"
3,Customer_004,210.13
4,Customer_005,"1,335.39"
5,Customer_006,"1,276.68"
6,Customer_007,"1,670.35"
7,Customer_008,"1,623.64"
8,Customer_009,-256.19
9,Customer_010,"1,290.92"


## 5. Choose a product or customer

The functions below return the amount or show the details behind it. Names can be entered in upper or lower case.

In [6]:
def price_impact_details(product=None, customer=None):
    result = comparison
    if product is not None:
        name = str(product).strip().lower()
        result = result[result['Product'].str.lower() == name]
    if customer is not None:
        name = str(customer).strip().lower()
        result = result[result['Customer'].str.lower() == name]
    if result.empty:
        raise ValueError('No data found. Check the product or customer name.')
    return result[detail_columns].copy()


def price_impact(product=None, customer=None):
    result = price_impact_details(product, customer)
    return round(result['Price_Impact'].sum(), 2)


def show_price_impact(product=None, customer=None):
    print(f'Price impact: EUR {price_impact(product, customer):,.2f}')
    if product is None and customer is None:
        display(product_summary)
    else:
        display(price_impact_details(product, customer))


### Product

Change the name and run the cell.

In [7]:
selected_product = "Product_A"
show_price_impact(product=selected_product)

Price impact: EUR 5,706.92


,Customer,Product,Quantity_Previous,Price_Previous,Price_Current,Price_Change,Price_Impact
0,Customer_001,Product_A,272,4.60,4.60,0.00,0.00
5,Customer_002,Product_A,479,5.20,5.36,0.16,76.64
10,Customer_003,Product_A,297,4.44,4.44,0.00,0.00
15,Customer_004,Product_A,141,4.90,5.15,0.25,35.25
20,Customer_005,Product_A,283,4.90,4.90,0.00,0.00
...,...,...,...,...,...,...,...
475,Customer_096,Product_A,241,4.70,4.70,0.00,0.00
480,Customer_097,Product_A,319,4.65,4.65,0.00,0.00
485,Customer_098,Product_A,462,4.61,4.75,0.14,64.68
490,Customer_099,Product_A,512,5.00,5.00,0.00,0.00


### Customer

The table shows the five product contributions that make up the customer total.

In [8]:
selected_customer = "Customer_001"
show_price_impact(customer=selected_customer)

Price impact: EUR 919.56


,Customer,Product,Quantity_Previous,Price_Previous,Price_Current,Price_Change,Price_Impact
0,Customer_001,Product_A,272,4.60,4.60,0.00,0.00
1,Customer_001,Product_B,243,7.58,6.97,-0.61,-148.23
2,Customer_001,Product_C,76,11.04,11.81,0.77,58.52
3,Customer_001,Product_D,406,16.89,15.88,-1.01,-410.06
4,Customer_001,Product_E,363,23.00,26.91,3.91,"1,419.33"


### Overall total

Without a filter, the function adds all customer-product impacts. Supply both filters to look at a single pair.

In [9]:
show_price_impact()

# Example for one pair:
# show_price_impact(product="Product_B", customer="Customer_001")

Price impact: EUR 143,231.87


,Product,Price_Impact
0,Product_A,"5,706.92"
1,Product_B,"-8,227.50"
2,Product_C,"51,566.61"
3,Product_D,"4,994.35"
4,Product_E,"89,191.49"


## 6. Use dropdowns

Run this cell, then choose a product or customer. Leave both at All for the total. Choosing both gives the result for one pair. The result updates when the selection changes. If your notebook viewer does not support widgets, use the example cells above.

In [10]:
try:
    import ipywidgets as widgets
except ImportError:
    print('Use the example cells above if widgets are not installed.')
else:
    widgets.interact(
        show_price_impact,
        product=widgets.Dropdown(
            options=[('All products', None)] + [(p, p) for p in product_summary['Product']]
        ),
        customer=widgets.Dropdown(
            options=[('All customers', None)] + [(c, c) for c in customer_summary['Customer']]
        )
    )


interactive(children=(Dropdown(description='product', options=(('All products', None), ('Product_A', 'Product_…

## 7. Check the totals

Check that both summaries add up to the overall result. Also check the price impact against sales repriced at the current price and old quantity.

In [11]:
assert round(product_summary['Price_Impact'].sum(), 2) == total_price_impact
assert round(customer_summary['Price_Impact'].sum(), 2) == total_price_impact

# Repricing the old quantity should give the same price impact.
repriced_sales = comparison['Price_Current'] * comparison['Quantity_Previous']
assert ((repriced_sales - comparison['Sales_Previous']).round(2)
        == comparison['Price_Impact']).all()
print('Product totals and customer totals agree with the overall result.')


Product totals and customer totals agree with the overall result.
